# Appendix A1: Chunking strategy study

Not one of "the 10 patterns" -- no mandatory 8-section template (same exemption as `00_baseline_no_rag.ipynb`/
`00b_long_context_baseline.ipynb`). Holds retrieval constant (hybrid + cross-encoder rerank,
`recipes/hybrid_rerank.py`) and varies only how the corpus is chunked, reporting a paper-level
hit@10 (`evals.metrics.paper_hit_at_k`) per variant.

**Requires the `corpus-build` extra** (`uv sync --extra corpus-build`) for `tiktoken`, unlike the 10
pattern notebooks which need only the base `uv sync` -- this is the one appendix that needs a real
tokenizer to build token-sized chunks.

**This committed run is real.** All 6 chunking variants use real `text-embedding-3-small`
embeddings (and, for the semantic-chunking variant, real embeddings to find its split points
too) against this repo's real, full paper text (`corpus/corpus_fulltext.jsonl`, fetched once via
the one-off `corpus/build_fulltext.py`), using real chunking logic
(`corpus/chunking_strategies.py`), with **zero live network calls at notebook-execution time**
for the chunking step itself. Running this notebook without `OPENAI_API_KEY` set
(`RAG_RECIPES_LLM=mock`) scores every row under `MockEmbedder`'s non-semantic hash vectors
instead, which only proves the code path runs, not real retrieval quality.

Note on ground truth: `qa_set.jsonl`'s `relevant_chunk_ids` reference the ORIGINAL corpus's
chunk_ids, which don't exist in any re-chunked variant. `paper_hit_at_k` compares at the PAPER
level instead (`paper_id` is stable across every chunking strategy) -- coarser than the main
leaderboard's `hit@10`, and stated here explicitly rather than silently swapped in.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from corpus.chunking_strategies import (
    chunk_document_aware,
    chunk_fixed,
    chunk_late,
    chunk_semantic,
)
from evals.metrics import bootstrap_ci, paper_hit_at_k
from evals.run import load_corpus_by_id, load_qa_set
from recipes.embeddings import get_embedder
from recipes.hybrid_rerank import build_retriever

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
embedder = get_embedder()
_IS_REAL_RUN = os.environ.get("RAG_RECIPES_LLM", "openai").lower() != "mock"

full_text_by_paper = {}
with open("../corpus/corpus_fulltext.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        full_text_by_paper[rec["paper_id"]] = rec["full_text"]

def _get_tokenizer():
    import tiktoken
    try:
        return tiktoken.encoding_for_model("gpt-4.1-mini")
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")

tokenizer = _get_tokenizer()

# Map each question to its relevant PAPER ids (not chunk ids) -- see the
# top-of-notebook note on why paper_hit_at_k exists.
relevant_paper_ids_by_qid = {
    r["qid"]: {corpus_by_id[cid]["paper_id"] for cid in r["relevant_chunk_ids"] if cid in corpus_by_id}
    for r in qa_set
}

K = 10


In [3]:
def print_results_table(rows):
    """rows: list of (label, ConfidenceInterval, is_real: bool)."""
    print(f"{'variant':<28} {'paper_hit@10':<24} {'status'}")
    for label, ci, is_real in rows:
        ci_str = f"{ci.mean:.3f}  [95% CI {ci.lower:.3f}, {ci.upper:.3f}]"
        status = "REAL" if is_real else "PENDING (mock)"
        print(f"{label:<28} {ci_str:<24} {status}")


## Build each chunking variant's corpus, then score paper_hit@10

In [4]:
def score_variant(variant_corpus_by_id, label):
    retrieve = build_retriever(variant_corpus_by_id, embedder=embedder)
    scores = []
    for r in qa_set:
        relevant_papers = relevant_paper_ids_by_qid[r["qid"]]
        if not relevant_papers:
            continue
        retrieved_ids = retrieve(r["question"], K)
        scores.append(paper_hit_at_k(retrieved_ids, relevant_papers, variant_corpus_by_id, K))
    return bootstrap_ci(scores)

def build_variant_corpus(chunks_per_paper: dict[str, list[dict]]) -> dict[str, dict]:
    """chunks_per_paper: paper_id -> list of {"text", "embed_text"} dicts.
    Returns a corpus_by_id-shaped dict; "text" is what gets embedded AND
    cited (embed_text is only used internally by chunk_late's caller, see
    below), chunk_id is synthesized as f"{paper_id}#{i}".
    """
    out = {}
    for paper_id, chunks in chunks_per_paper.items():
        for i, chunk in enumerate(chunks):
            cid = f"{paper_id}#{i}"
            out[cid] = {"chunk_id": cid, "paper_id": paper_id, "text": chunk["embed_text"]}
    return out

results = []


### fixed-256 / fixed-512 / fixed-1024

In [5]:
for chunk_tokens in (256, 512, 1024):
    chunks_per_paper = {
        paper_id: chunk_fixed(text, chunk_tokens=chunk_tokens, tokenizer=tokenizer)
        for paper_id, text in full_text_by_paper.items()
    }
    variant_corpus = build_variant_corpus(chunks_per_paper)
    ci = score_variant(variant_corpus, f"fixed-{chunk_tokens}")
    results.append((f"fixed-{chunk_tokens}", ci, _IS_REAL_RUN))


E:\Rajesh\PycharmProjects\rag-recipes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6219.43it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6100.84it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4425.48it/s]

### semantic (embeddings-based)

In [6]:
chunks_per_paper = {
    paper_id: chunk_semantic(text, embedder=embedder, embedding_model="text-embedding-3-small", tokenizer=tokenizer)
    for paper_id, text in full_text_by_paper.items()
}
variant_corpus = build_variant_corpus(chunks_per_paper)
ci = score_variant(variant_corpus, "semantic")
results.append(("semantic", ci, _IS_REAL_RUN))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4704.50it/s]

### document-aware (section-boundary)

In [7]:
chunks_by_paper_original = {}
for chunk in corpus_by_id.values():
    chunks_by_paper_original.setdefault(chunk["paper_id"], []).append(chunk)

chunks_per_paper = {
    paper_id: chunk_document_aware(chunks) for paper_id, chunks in chunks_by_paper_original.items()
}
variant_corpus = build_variant_corpus(chunks_per_paper)
ci = score_variant(variant_corpus, "document-aware")
results.append(("document-aware", ci, _IS_REAL_RUN))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:  26%|██▌       | 101/393 [00:00<00:00, 1007.77it/s]

Loading weights:  51%|█████▏    | 202/393 [00:00<00:00, 926.73it/s] 

Loading weights:  75%|███████▌  | 296/393 [00:00<00:00, 864.58it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 919.28it/s]

### late chunking (context-window approximation -- see top-of-notebook disclaimer)

In [8]:
chunks_per_paper = {
    paper_id: chunk_late(text, chunk_tokens=512, tokenizer=tokenizer)
    for paper_id, text in full_text_by_paper.items()
}
variant_corpus = build_variant_corpus(chunks_per_paper)
ci = score_variant(variant_corpus, "late (approx.)")
results.append(("late (approx.)", ci, _IS_REAL_RUN))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:  30%|███       | 119/393 [00:00<00:00, 1179.64it/s]

Loading weights:  63%|██████▎   | 247/393 [00:00<00:00, 1232.48it/s]

Loading weights:  94%|█████████▍| 371/393 [00:00<00:00, 1210.16it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1208.25it/s]

## Results

In [9]:
print_results_table(results)

variant                      paper_hit@10             status
fixed-256                    1.000  [95% CI 1.000, 1.000] REAL
fixed-512                    1.000  [95% CI 1.000, 1.000] REAL
fixed-1024                   1.000  [95% CI 1.000, 1.000] REAL
semantic                     1.000  [95% CI 1.000, 1.000] REAL
document-aware               1.000  [95% CI 1.000, 1.000] REAL
late (approx.)               1.000  [95% CI 1.000, 1.000] REAL


## Findings

All 6 chunking variants score `paper_hit@10` = 1.000 on this 18-paper pilot corpus and
18-question eval set -- chunking strategy makes no visible difference at this scale. That is
a real, honest finding, not a placeholder: a small enough corpus makes almost any reasonable
chunking strategy find the right paper. This comparison is more likely to differentiate
strategies once run against a larger corpus (~300 chunks), where paper-level
hit@10 has more room to vary.
